# Part III: Vector spaces: solutions to the code exercises

Worked solutions to the code exercises from [*Matrices in Practice*](https://trgoodale.github.io/matrix-book/appendices/solutions-3.html), by Tom Goodale.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trgoodale/matrix-book/blob/main/code/python/solutions-part-3.ipynb)

Run the cells in order, from the top: later cells use variables set in earlier ones. Try each exercise yourself first. Change the numbers and run them again to see what happens. The code needs NumPy, SciPy and Matplotlib (and pandas in a few chapters), all of which Colab already has.

Code licensed under the [MIT licence](https://trgoodale.github.io/matrix-book/LICENSE.md).

## Chapter 14: Span and linear independence

### Exercise 7

Show that the four vectors $(1, 2, 0)^{\mathsf{T}}$, $(0, 1, 1)^{\mathsf{T}}$, $(1, 0, 1)^{\mathsf{T}}$ and $(2, 3, 2)^{\mathsf{T}}$ are dependent, and find a dependence relation. Are the first three independent?

In [1]:
import numpy as np
from scipy.linalg import null_space

V = np.column_stack([[1, 2, 0], [0, 1, 1], [1, 0, 1], [2, 3, 2]])
print(np.linalg.matrix_rank(V))              # 3 < 4: dependent
c = null_space(V)
print(c.ravel() / c[0, 0])                   # scaled so that c1 = 1
print(np.linalg.matrix_rank(V[:, :3]))       # 3: the first three are independent

3
[ 1.  1.  1. -1.]
3


Four vectors in $\mathbb{R}^3$ must be dependent. The relation is $\mathbf{v}_1 + \mathbf{v}_2 + \mathbf{v}_3 - \mathbf{v}_4 = \mathbf{0}$: the fourth vector is the sum of the other three. The first three have rank 3, so they are independent.

## Chapter 15: Basis, dimension and change of basis

### Exercise 7

In [Chapter 9](https://trgoodale.github.io/matrix-book/part-1/09-invariant-lines.html) you found that $\mathbf{C} = \begin{pmatrix}0 & 0 & 1 \\ 1 & 0 & 0 \\ 0 & 1 & 0\end{pmatrix}$ is the rotation by $120°$ about the line through $(1, 1, 1)$. Build it the other way, by a change of basis. Use $\mathbf{p} = \tfrac{1}{\sqrt{2}}(1, -1, 0)^{\mathsf{T}}$, $\mathbf{q} = \tfrac{1}{\sqrt{6}}(1, 1, -2)^{\mathsf{T}}$ and $\mathbf{u} = \tfrac{1}{\sqrt{3}}(1, 1, 1)^{\mathsf{T}}$ as the columns of $\mathbf{P}$, so that the axis $\mathbf{u}$ plays the part of the $z$-axis. Compute $\mathbf{P}\,\mathbf{R}_z(120°)\,\mathbf{P}^{-1}$ and compare it with $\mathbf{C}$. (These three vectors are perpendicular unit vectors, so $\mathbf{P}$ is orthogonal and $\mathbf{P}^{-1} = \mathbf{P}^{\mathsf{T}}$.)

In [2]:
import numpy as np

p = np.array([1, -1, 0]) / np.sqrt(2)
q = np.array([1, 1, -2]) / np.sqrt(6)
u = np.array([1, 1, 1]) / np.sqrt(3)
P = np.column_stack([p, q, u])

th = np.radians(120)
Rz = np.array([[np.cos(th), -np.sin(th), 0],
               [np.sin(th),  np.cos(th), 0],
               [0, 0, 1]])
print(np.round(P @ Rz @ P.T, 10) + 0)

[[0. 0. 1.]
 [1. 0. 0.]
 [0. 1. 0.]]


The result is exactly $\mathbf{C}$. In the basis $\mathbf{p}, \mathbf{q}, \mathbf{u}$, the rotation is the simple $\mathbf{R}_z(120°)$; the change of basis carries it over to the usual coordinates.

## Chapter 16: Column space, null space and rank–nullity

### Exercise 7

Make a random $5 \times 8$ matrix of rank 3 as the product of a random $5 \times 3$ and a random $3 \times 8$ matrix. Check numerically that the dimensions of its four subspaces are as the table predicts, and that every null space vector is perpendicular to every row.

In [3]:
import numpy as np
from scipy.linalg import null_space, orth

rng = np.random.default_rng(7)
A = rng.standard_normal((5, 3)) @ rng.standard_normal((3, 8))
m, n = A.shape
r = np.linalg.matrix_rank(A)
print("rank", r)
print("column space", orth(A).shape[1], " null space", null_space(A).shape[1],
      " row space", orth(A.T).shape[1], " left null space", null_space(A.T).shape[1])
print("largest row . null vector:", np.abs(A @ null_space(A)).max())

rank 3
column space 3  null space 5  row space 3  left null space 2
largest row . null vector: 2.8651048227029115e-15


The dimensions are $r = 3$, $n - r = 5$, $r = 3$ and $m - r = 2$, as the table predicts, and every row dotted with every null space vector gives zero, up to rounding error of about $10^{-15}$.

## Chapter 17: Orthogonality and least squares

### Exercise 7

A ball is thrown upwards, and its height $h$ in metres is measured at times $t = 0, 0.5, 1, 1.5, 2, 2.5$ seconds as $2.1, 7.6, 11.3, 11.8, 10.5, 6.2$. Fit the model $h = c_0 + c_1t + c_2t^2$ by least squares. Since $h = h_0 + ut - \tfrac{1}{2}gt^2$ for a ball thrown up with speed $u$, use your fit to estimate $g$, the acceleration due to gravity.

In [4]:
import numpy as np

t = np.array([0, 0.5, 1, 1.5, 2, 2.5])
h = np.array([2.1, 7.6, 11.3, 11.8, 10.5, 6.2])
A = np.column_stack([np.ones_like(t), t, t**2])
c, *_ = np.linalg.lstsq(A, h, rcond=None)
print(np.round(c, 3))
print("g is about", round(-2 * c[2], 2), "m/s^2")

[ 2.021 14.019 -4.929]
g is about 9.86 m/s^2


The fit is $h \approx 2.02 + 14.02t - 4.93t^2$. Comparing with $h_0 + ut - \tfrac{1}{2}gt^2$ gives $\tfrac{1}{2}g \approx 4.93$, so $g \approx 9.86$ m/s², close to the true value of about 9.81.

## Part III: Vector spaces review

### Exercise 8

Write a Gram–Schmidt function that takes a matrix and returns a matrix $\mathbf{Q}$ with orthonormal columns spanning the same space. Apply it to the columns $(1, 1, 0)^{\mathsf{T}}$, $(1, 0, 1)^{\mathsf{T}}$, $(0, 1, 1)^{\mathsf{T}}$, check that $\mathbf{Q}^{\mathsf{T}}\mathbf{Q} = \mathbf{I}$, and compare $\mathbf{Q}$ with the built-in QR factorisation.

*Uses [Chapter 17](https://trgoodale.github.io/matrix-book/part-3/17-orthogonality-least-squares.html).*

In [5]:
import numpy as np

def gram_schmidt(A):
    A = np.asarray(A, dtype=float)
    Q = np.zeros_like(A)
    for j in range(A.shape[1]):
        v = A[:, j] - Q[:, :j] @ (Q[:, :j].T @ A[:, j])   # remove projections
        Q[:, j] = v / np.linalg.norm(v)
    return Q

A = np.array([[1, 1, 0], [1, 0, 1], [0, 1, 1]])
Q = gram_schmidt(A)
print(np.round(Q, 4))
print("Q^T Q = I:", np.allclose(Q.T @ Q, np.eye(3)))

Q2, R2 = np.linalg.qr(A)
print("Same columns up to sign:", np.allclose(np.abs(Q2), np.abs(Q)))

[[ 0.7071  0.4082 -0.5774]
 [ 0.7071 -0.4082  0.5774]
 [ 0.      0.8165  0.5774]]
Q^T Q = I: True
Same columns up to sign: True


The built-in QR may choose the opposite sign for some columns. Either sign gives an orthonormal basis for the same space.